# RabTech Academy — Task 2: Customer Churn Responsible Baseline

This notebook uses the **starter CSV supplied for the task**.

Goals:
1. Inspect and document the data.
2. Establish a non-ML baseline.
3. Train a simple, interpretable ML baseline.
4. Evaluate accuracy, precision, recall, F1 and the confusion matrix.
5. Record limitations and responsible-use safeguards.

**Important:** This dataset has only 12 rows, so the results are educational and should not be treated as production evidence.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

CSV_PATH = Path("customer-churn-training.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Put the CSV in the same folder as this notebook."
    )

df = pd.read_csv(CSV_PATH)
df


## 1. Basic data inspection

The starter dataset should be checked before modeling.


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["churned"].value_counts().sort_index())

print("\nFirst rows:")
display(df.head())


## 2. Data quality checks

For this supplied dataset we expect:
- 12 rows
- 7 columns
- no missing values
- no duplicate rows
- binary target `churned`


In [ ]:
assert df.shape == (12, 7), f"Unexpected shape: {df.shape}"
assert df.isna().sum().sum() == 0, "Missing values were found."
assert df.duplicated().sum() == 0, "Duplicate rows were found."
assert set(df["churned"].unique()) == {0, 1}, "Target is not binary as expected."

print("All basic quality assertions passed.")

print("\nTarget counts:")
target_counts = df["churned"].value_counts().sort_index()
print(target_counts)

print("\nTarget percentages:")
print((target_counts / len(df) * 100).round(1))


## 3. Feature review and leakage control

`customer_id` is an identifier, so it is not used as a predictive feature.

`churned` is the target and is also excluded from the input features.

The remaining columns are used as candidate predictors.


In [ ]:
target = "churned"
id_column = "customer_id"

feature_columns = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days",
    "plan_type",
]

X = df[feature_columns].copy()
y = df[target].copy()

print("Features used:")
print(X.columns.tolist())
print("\nTarget:", target)


## 4. Non-ML baseline

A simple baseline predicts the majority class for every customer.

The majority class in this dataset is `0` (did not churn).


In [ ]:
majority_class = int(y.mode()[0])
baseline_predictions = np.full(len(y), majority_class)

baseline_accuracy = accuracy_score(y, baseline_predictions)

print("Majority class:", majority_class)
print(f"Baseline accuracy: {baseline_accuracy:.3f} ({baseline_accuracy*100:.1f}%)")


## 5. Prepare the ML pipeline

A Logistic Regression model is used because it is simple and interpretable.

- Numeric columns are standardized.
- `plan_type` is one-hot encoded.
- The transformation and model are kept inside one pipeline.


In [ ]:
numeric_features = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days",
]

categorical_features = ["plan_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

print(model)


## 6. Cross-validated ML evaluation

Because there are only 12 observations, a single train/test split would give a very unstable estimate.

We therefore use stratified 5-fold cross-validation. This keeps the class distribution as balanced as possible across folds.

The reported metrics should still be treated cautiously because the dataset is extremely small.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_predictions = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    method="predict"
)

accuracy = accuracy_score(y, cv_predictions)
precision = precision_score(y, cv_predictions, zero_division=0)
recall = recall_score(y, cv_predictions, zero_division=0)
f1 = f1_score(y, cv_predictions, zero_division=0)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")

print("\nClassification report:")
print(classification_report(y, cv_predictions, zero_division=0))


## 7. Confusion matrix

For churn prediction:

- True Positive: correctly identified churner.
- False Positive: customer flagged as churn risk but did not churn.
- False Negative: customer churned but was not flagged.
- True Negative: correctly identified non-churner.


In [ ]:
cm = confusion_matrix(y, cv_predictions, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0 (No churn)", "Actual 1 (Churn)"],
    columns=["Predicted 0", "Predicted 1"],
)

display(cm_df)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks([0, 1], labels=["Predicted 0", "Predicted 1"])
ax.set_yticks([0, 1], labels=["Actual 0", "Actual 1"])
ax.set_xlabel("Prediction")
ax.set_ylabel("Actual")
ax.set_title("Cross-validated confusion matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 8. Compare ML with the non-ML baseline

The important question is not simply "Is the ML accuracy high?"

The first question is whether the ML approach provides useful evidence beyond the simple majority-class baseline.


In [ ]:
comparison = pd.DataFrame({
    "Approach": ["Majority-class baseline", "Logistic Regression"],
    "Accuracy": [baseline_accuracy, accuracy],
    "Precision": [np.nan, precision],
    "Recall": [np.nan, recall],
    "F1": [np.nan, f1],
})

display(comparison)


## 9. Simple data exploration

These summaries are descriptive only. With 12 rows, they should not be treated as general customer behavior.

The observed averages can help us inspect whether the labels appear associated with the available features.


In [ ]:
numeric_summary = df.groupby("churned")[
    ["tenure_months", "support_tickets", "monthly_spend_inr", "last_login_days"]
].mean().round(2)

display(numeric_summary)

print("\nPlan type vs churn:")
display(pd.crosstab(df["plan_type"], df["churned"]))


## 10. Final model fit for demonstration

After evaluation, we fit the pipeline on the complete starter dataset only to demonstrate how a future prediction could be generated.

This is **not** a production deployment model.


In [ ]:
model.fit(X, y)

# Example: use the first customer as a demonstration input.
example_customer = X.iloc[[0]]

example_probability = model.predict_proba(example_customer)[0, 1]
example_prediction = model.predict(example_customer)[0]

print("Example customer:")
display(example_customer)

print(f"Predicted churn probability: {example_probability:.3f}")
print(f"Predicted class: {example_prediction}")


## 11. Abstention / human-review concept

A model should not be treated as certain.

For a real system, an operational threshold should be selected using validation data and business costs. If the prediction is too uncertain, the system should abstain and send the case to human review.

The starter dataset is too small to justify a final probability threshold, so this notebook does not claim one is production-ready.


In [ ]:
probabilities = model.predict_proba(X)[:, 1]

review_demo = pd.DataFrame({
    "customer_id": df["customer_id"],
    "churn_probability": probabilities.round(3),
})

# Illustrative review flag only — NOT a validated production threshold.
review_demo["illustrative_human_review"] = (
    (review_demo["churn_probability"] >= 0.40)
    & (review_demo["churn_probability"] <= 0.60)
)

display(review_demo)
print(
    "The 0.40–0.60 range above is illustrative only; "
    "it must be validated before real use."
)


## 12. Responsible-use conclusions

### What this exercise demonstrates
- The decision and target were defined before modeling.
- An identifier was excluded from the feature set.
- Data quality checks were performed.
- A non-ML baseline was established.
- A simple ML model was evaluated.
- False-positive and false-negative risks were considered.
- Human review and rollback were defined as safeguards.

### What this exercise does NOT demonstrate
- Production-level accuracy.
- Generalization to a real customer population.
- Fairness across demographic groups.
- Valid probability calibration.
- A final business threshold.
- Real-world consent/licensing compliance.

The starter dataset has only 12 rows, so these limitations are central to the result.


## 13. Submission checklist

Before submitting Task 2, confirm that you have:

- [x] ML problem-framing memo
- [x] Responsible data card
- [x] Baseline notebook
- [x] Risk register
- [x] Used the RabTech starter CSV
- [x] Documented the dataset limitations
- [x] Avoided claiming production readiness
